# 单端口分层校准

## 简介

如果待测设备是互易的，则可以使用单端口网络分析仪来测量一个两端口设备。这是通过执行两次校准来实现的，这就是它被称为“分层”校准的原因。首先，像往常一样对网络分析仪的测试端口进行校准。这被称为“第一层”。接下来，将设备连接到测试端口，并在设备的一端进行校准，这被称为“第二层”。下图显示了一个示意图：![方框图](oneport_tiered_calibration/images/boxDiagram.svg)

这个 notebook 将演示如何使用 [skrf](http://scikit-rf.org) 进行两级单端口校准。我们将使用采集到的数据来表征一个波导到共面波导（CPW）探头。因此，对于这个具体的例子，上面的示意图如下所示：![探头示意图](oneport_tiered_calibration/images/probe.svg)

## 一些数据

可用的数据位于文件夹 `'tier1/'` 和 `'tier2/'` 中。

In [ ]:
!ls {"oneport_tiered_calibration/"}

（如果您没有这些示例的 git 仓库，则可以在 [此处](https://github.com/scikit-rf/scikit-rf/tree/master/doc/source/examples/metrology/oneport_tiered_calibration) 找到此笔记本的数据。）在每个文件夹中，您将找到两个子文件夹，分别名为“`ideals/`”和“`measured/`”。它们分别包含校准标准的理想响应和测量响应的 Touchstone 文件。

In [ ]:
!ls {"oneport_tiered_calibration/tier1/"}

第一层位于波导接口处，包含以下一组校准标准：* 短路* 延迟短路* 负载* 辐射开路（实际上是一个开放的波导）

In [ ]:
!ls {"oneport_tiered_calibration/tier1/measured/"}

## 创建校准

### 第一层级

首先定义第一级的校准。

In [ ]:
import matplotlib.pyplot as plt

import skrf as rf
from skrf.calibration import OnePort
from skrf.io.general import read_all_networks

%matplotlib inline

rf.stylely()


tier1_ideals = read_all_networks('oneport_tiered_calibration/tier1/ideals/')
tier1_measured = read_all_networks('oneport_tiered_calibration/tier1/measured/')


tier1 = OnePort(measured = tier1_measured,
                ideals = tier1_ideals,
                name = 'tier1',
                sloppy_input=True)
tier1

因为我们以相同的名称保存了对应的*理想*和*测量*标准，所以校准过程会在初始化时自动对齐这些标准。（有关创建校准对象的更多信息，请参见[文档](../../tutorials/Calibration.ipynb)）。同样适用于第二级的校准。

### 第二级

In [ ]:
tier2_ideals = read_all_networks('oneport_tiered_calibration/tier2/ideals/')
tier2_measured = read_all_networks('oneport_tiered_calibration/tier2/measured/')


tier2 = OnePort(measured = tier2_measured,
                ideals = tier2_ideals,
                name = 'tier2',
                sloppy_input=True)
tier2

## 误差网络

每个单端口校准都包含一个两端口误差网络，该网络由计算出的误差系数确定。*tier1* 模型中的误差网络代表 VNA，而 *tier2* 模型中的误差网络代表 VNA **和** DUT。可以通过参数 `'error_ntwk'` 对它们进行可视化。对于 tier 1，

In [ ]:
tier1.error_ntwk.plot_s_db()
plt.title('Tier 1 Error Network')

同样适用于第二层。

In [ ]:
tier2.error_ntwk.plot_s_db()
plt.title('Tier 2 Error Network')

## 从DUT中去除寄生效应

如前所述，*tier1* 模型中的误差网络代表 VNA，而 *tier2* 模型中的误差网络代表 VNA+DUT。因此，为了确定 DUT 的响应，我们将 VNA 的逆 S 散射参数与 VNA+DUT 串联。$$ DUT = VNA^{-1}\cdot (VNA \cdot DUT)$$在 skrf 中，操作如下：

In [ ]:
dut = tier1.error_ntwk.inv ** tier2.error_ntwk
dut.name = 'probe'
dut.plot_s_db()
plt.title('Probe S-parameters')
plt.ylim(-60,10)

您可能希望将其保存到磁盘，以备将来使用：```pythondut.write_touchstone()```

In [ ]:
!ls {"probe*"}